# Module 4 — RAG Pipeline

Retrieval-Augmented Generation pipeline for mental health support.
Given a user question, the system retrieves relevant counseling conversations
from a Qdrant vector database and generates an empathetic response using an LLM.

Dataset: Amod/mental_health_counseling_conversations
Vector DB: Qdrant Cloud
Embeddings: sentence-transformers
LLM: Groq API

**Before running this notebook, make sure you have installed dependencies:**
pip install -r requirements.txt

# Imports and Checking Environment

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from groq import Groq

load_dotenv()

# Verify everything loaded — if any print None, check your .env file
print("GROQ key loaded:   ", "yes" if os.getenv("GROQ_API_KEY") else "NO — CHECK .env")
print("Qdrant URL loaded: ", "yes" if os.getenv("QDRANT_URL") else "NO — CHECK .env")
print("Qdrant key loaded: ", "yes" if os.getenv("QDRANT_API_KEY") else "NO — CHECK .env")

e:\NLP\Mental-Health-RAG-Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GROQ key loaded:    yes
Qdrant URL loaded:  yes
Qdrant key loaded:  yes


# Load and Explore the Dataset

In [3]:
from tqdm.auto import tqdm

dataset = load_dataset("Amod/mental_health_counseling_conversations")
df = pd.DataFrame(dataset['train'])

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst example:")
print(f"CONTEXT:\n{df['Context'][0]}\n")
print(f"RESPONSE:\n{df['Response'][0]}")

e:\NLP\Mental-Health-RAG-Chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mohamed\.cache\huggingface\hub\datasets--Amod--mental_health_counseling_conversations. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 3512/3512 [00:00<00:00, 65583.56 examples/

Shape: (3512, 2)
Columns: ['Context', 'Response']

First example:
CONTEXT:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?

RESPONSE:
If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone d

In [ ]:
# check for missing values
df.isnull().sum()

Context     0
Response    0
dtype: int64

In [5]:
# Combine context and response into a single document for embedding
df['document'] = "User: " + df['Context'] + "\nCounselor: " + df['Response']
# Preview the combined document
print(f"\nSample document:\n{df['document'][0][:300]}...")



Sample document:
User: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeli...
